# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [10]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [11]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [12]:
missing_values = spaceship.isnull().sum()

# Print the results
print(missing_values)

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


In [13]:
df = spaceship.dropna()


print(df.shape)

(6606, 14)


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Assuming 'spaceship' is your DataFrame
df = spaceship

# Identify categorical columns including 'object' and 'bool' types
categorical_columns = df.select_dtypes(include=['object', 'bool']).columns

# Make sure 'Transported' is your target and not included in categorical encode list
if 'Transported' in categorical_columns:
    categorical_columns = categorical_columns.drop('Transported')

# One-hot encode the categorical columns
df_encoded = pd.get_dummies(df, columns=categorical_columns, drop_first=True)

# Define the features and target
features = df_encoded.drop(columns=['Transported'])
target = df_encoded['Transported']



**Perform Train Test Split**

In [15]:
# Perform the train-test split
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# Initialize and fit the StandardScaler to the training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("DataFrame 'spaceship' has been processed.")
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

DataFrame 'spaceship' has been processed.
X_train_scaled shape: (6954, 23735)
X_test_scaled shape: (1739, 23735)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import BaggingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.impute import SimpleImputer

# Dummy data for example; adjust as needed to fit your actual DataFrame
# df = pd.DataFrame(...)  # Load your dataframe

# Ensure you have defined 'features' and 'target'
# Assuming 'Transported' is the target
features = df.drop(columns=['Transported'])
target = df['Transported']

# Split your data
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# One-Hot encode categorical variables
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

# Align the training and test data to ensure they have the same columns
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# Handle missing values using SimpleImputer on numeric data only
numeric_cols = X_train_encoded.select_dtypes(include=[np.number]).columns

imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train_encoded[numeric_cols])
X_test_imputed = imputer.transform(X_test_encoded[numeric_cols])

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Instantiate KNeighborsRegressor for the base model
base_model = KNeighborsRegressor()

# Bagging with the KNeighborsRegressor
bagging_model = BaggingRegressor(
    estimator=base_model,
    n_estimators=50,  # You can adjust this
    max_samples=0.5,
    bootstrap=True,
    random_state=42
)

# Fit Bagging model
bagging_model.fit(X_train_scaled, y_train)

# Make predictions and evaluate
y_pred_bagging = bagging_model.predict(X_test_scaled)
mse_bagging = mean_squared_error(y_test, y_pred_bagging)
print(f"Bagging KNeighborsRegressor Mean Squared Error (MSE): {mse_bagging}")

Bagging KNeighborsRegressor Mean Squared Error (MSE): 0.17365429327199539


- Random Forests

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

# Assume df is your DataFrame and 'Transported' is your target column
features = df.drop(columns=['Transported'])
target = df['Transported']

# 1. Split your data
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# 2. One-Hot encode categorical variables
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

# Align training and testing data
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# 3. Handle missing values using SimpleImputer
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train_encoded)
X_test_imputed = imputer.transform(X_test_encoded)

# 4. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# 5. Instantiate RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# 6. Fit and Evaluate
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
mse_rf = mean_squared_error(y_test, y_pred_rf)
print(f"Random Forest Mean Squared Error (MSE): {mse_rf}")

KeyboardInterrupt: 

- Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer


X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy='mean')

# Fit imputer on training data and transform both train and test sets
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Initialize GradientBoostingRegressor
gb_model = GradientBoostingRegressor(
    n_estimators=100,          # Number of boosting stages
    learning_rate=0.1,         # Impact of each tree
    max_depth=3,               # Depth of each tree
    random_state=42
)

# Fit the Gradient Boosting model
gb_model.fit(X_train_scaled, y_train)

# Predict and evaluate
y_pred_gb = gb_model.predict(X_test_scaled)
mse_gb = mean_squared_error(y_test, y_pred_gb)
print(f"Gradient Boosting Mean Squared Error (MSE): {mse_gb}")


Gradient Boosting Mean Squared Error (MSE): 0.14605264135991813


- Adaptive Boosting

In [9]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer

# 1. Split your data
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# 2. Handle missing values (Imputation in one step)
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# 3. Initialize AdaBoost Regressor
ada_model = AdaBoostRegressor(n_estimators=100, learning_rate=1.0, random_state=42)

# 4. Fit the model
ada_model.fit(X_train, y_train)

# 5. Predict and evaluate
y_pred_ada = ada_model.predict(X_test)
mse_ada = mean_squared_error(y_test, y_pred_ada)
print(f"AdaBoost Mean Squared Error (MSE): {mse_ada}")

KeyboardInterrupt: 

Which model is the best and why?

Gradient Boosting is Better:
Lower MSE: The Gradient Boosting model has a lower MSE, indicating that it makes smaller average errors in predictions than the Bagging KNeighborsRegressor.